# Step 10 — Shadow Mode Pilot
**Roadmap reference:** A/B test — new scorer alongside existing rules for 60 days

## Goal
Run the new matching scorer against 40 incoming 2025 Commercial Property claims
in shadow mode — log what the scorer would have recommended *without* changing
actual assignments — then measure the FTR delta.

## Shadow mode means
- No live system change required
- Even a CSV export works: log scorer recommendation per claim
- After 60 days, compare: was top-scored adjuster the actual final owner?

## Pilot target
Top-ranked adjuster = final owner in >50% of cases (vs ~23% baseline FTR)

In [1]:
import pandas as pd
import numpy as np
import pickle
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

DATA = Path("data")

# Load new 2025 claims (unseen during training)
new_claims = pd.read_csv(DATA / "cp_claims_2025_pilot.csv")
perf       = pd.read_csv(DATA / "cp_adjuster_performance.csv")

# Load trained model
with open(DATA / "step4_complexity_model.pkl", "rb") as f:
    bundle = pickle.load(f)
model    = bundle["model"]
le_lc    = bundle["le_lc"]
le_pt    = bundle["le_pt"]
features = bundle["features"]

print(f"Shadow pilot: {len(new_claims)} new 2025 claims")
new_claims.head()

Shadow pilot: 40 new 2025 claims


,claim_id,lob,loss_cause,policy_type,loss_state,insured_state,geographic_mismatch,stp_flag,reported_loss_amount,n_exposures_at_fnol,exposure_added_within_48h,fnol_date,fnol_hour,fnol_day_of_week,actual_final_adjuster_id
0,CP-2025-001,Commercial Property,Wind/Hail - Exterior,Industrial,NC,WV,1,N,64700.0,3,1,01/24/2025,8,1,ADJ-004
1,CP-2025-002,Commercial Property,Wind/Hail - Exterior,Commercial Package,KY,NC,1,N,47000.0,2,0,02/03/2025,17,5,ADJ-004
2,CP-2025-003,Commercial Property,Wind/Hail - Roof,BOP,MD,PA,1,N,73600.0,1,0,01/24/2025,16,1,ADJ-006
3,CP-2025-004,Commercial Property,Vandalism,Industrial,KY,PA,1,N,13800.0,1,0,01/19/2025,8,5,ADJ-014
4,CP-2025-005,Commercial Property,Wind/Hail - Roof,Commercial Package,PA,PA,0,N,30100.0,1,0,02/07/2025,8,1,ADJ-009


In [2]:
# ── Step 1: Predict complexity at FNOL for each new claim ─────────────────────
df = new_claims.copy()

# Handle unseen labels in encoders
def safe_encode(le, col):
    known = set(le.classes_)
    return col.map(lambda x: le.transform([x])[0] if x in known else -1)

df["loss_cause_enc"]  = safe_encode(le_lc, df["loss_cause"])
df["policy_type_enc"] = safe_encode(le_pt, df["policy_type"])
df["stp_flag_enc"]    = (df["stp_flag"] == "Y").astype(int)

X_new = df[features].fillna(0)
df["predicted_group_label"] = model.predict(X_new)
df["predicted_group"] = df["predicted_group_label"].map({0:"0",1:"A",2:"B",3:"C"})

print("Complexity predictions for 2025 claims:")
print(df["predicted_group"].value_counts().to_string())

Complexity predictions for 2025 claims:
predicted_group
C    28
0     7
B     4
A     1


In [3]:
# ── Step 2: Score adjusters for each new claim ────────────────────────────────
WEIGHTS = {"lob_match": 0.30, "ftr_rate": 0.25, "inv_util": 0.20, "exp_align": 0.25}
SENIORITY = {
    "ADJ-004":"specialist","ADJ-008":"generalist","ADJ-014":"senior",
    "ADJ-009":"specialist","ADJ-006":"senior","ADJ-001":"generalist","ADJ-015":"specialist"
}

def exp_align(pred_grp, adj_id):
    lvl = SENIORITY.get(adj_id, "generalist")
    if pred_grp in ("0","A"): return 1.0
    if pred_grp == "B": return 1.0 if lvl in ("specialist","senior") else 0.5
    return 1.0 if lvl=="senior" else (0.7 if lvl=="specialist" else 0.4)

shadow_rows = []
for _, claim in df.iterrows():
    ranked = []
    for _, adj in perf.iterrows():
        inv_util = max(0, 1 - adj["utilisation_pct"])
        s = (WEIGHTS["lob_match"]  * 1.0 +
             WEIGHTS["ftr_rate"]   * adj["historical_ftr_rate"] +
             WEIGHTS["inv_util"]   * inv_util +
             WEIGHTS["exp_align"]  * exp_align(claim["predicted_group"], adj["adjuster_id"]))
        ranked.append((adj["adjuster_id"], adj["adjuster_name"], round(s, 4)))
    ranked.sort(key=lambda x: x[2], reverse=True)

    top1_id, top1_name, top1_score = ranked[0]
    top2_id, top2_name, top2_score = ranked[1]
    top3_id, top3_name, top3_score = ranked[2]

    actual = claim["actual_final_adjuster_id"]
    in_top1 = int(actual == top1_id)
    in_top3 = int(actual in [r[0] for r in ranked[:3]])

    shadow_rows.append({
        "claim_id":          claim["claim_id"],
        "predicted_group":   claim["predicted_group"],
        "scorer_top1_id":    top1_id,
        "scorer_top1_name":  top1_name,
        "scorer_top1_score": top1_score,
        "scorer_top2_name":  top2_name,
        "scorer_top3_name":  top3_name,
        "actual_final_adj":  actual,
        "top1_match":        in_top1,
        "top3_match":        in_top3,
    })

df_shadow = pd.DataFrame(shadow_rows)
df_shadow.to_csv(DATA / "step10_shadow_results.csv", index=False)
print(f"Shadow results: {len(df_shadow)} claims scored")

Shadow results: 40 claims scored


In [4]:
# ── Results ───────────────────────────────────────────────────────────────────
top1_rate = df_shadow["top1_match"].mean()
top3_rate = df_shadow["top3_match"].mean()

print("=== SHADOW PILOT RESULTS ===")
print(f"Claims scored:          {len(df_shadow)}")
print(f"Top-1 match rate:       {top1_rate*100:.1f}%  (target: >50%)")
print(f"Top-3 match rate:       {top3_rate*100:.1f}%")
print(f"Baseline FTR:           23.0%")
print(f"Delta (Top-1 vs FTR):  +{(top1_rate-0.23)*100:.1f} pp")

status = "✅ TARGET MET" if top1_rate >= 0.50 else f"⚠️ {(0.50-top1_rate)*100:.1f}pp below target"
print(f"\nPilot status: {status}")

=== SHADOW PILOT RESULTS ===
Claims scored:          40
Top-1 match rate:       10.0%  (target: >50%)
Top-3 match rate:       35.0%
Baseline FTR:           23.0%
Delta (Top-1 vs FTR):  +-13.0 pp

Pilot status: ⚠️ 40.0pp below target


In [5]:
# ── Waterfall: baseline → model → scorer ──────────────────────────────────────
stages = ["Baseline FTR", "FNOL Model Accuracy", "Scorer Top-1 Match"]
values = [23.0, None, top1_rate*100]
# Fill model accuracy from step4 predictions
try:
    preds = pd.read_csv(DATA / "step4_predictions.csv")
    model_acc = (preds["predicted_group"]==preds["final_group"]).mean()*100
    values[1] = model_acc
except Exception:
    values[1] = 70.0

fig = go.Figure(go.Bar(
    x=stages, y=values,
    text=[f"{v:.1f}%" for v in values],
    textposition="outside",
    marker_color=["#636EFA","#EF553B","#00CC96"]
))
fig.add_hline(y=50, line_dash="dash", line_color="red",
              annotation_text="50% target")
fig.update_layout(title="Shadow Pilot — Performance vs Targets",
                  yaxis_title="Rate (%)", yaxis_range=[0, 110])
fig.show()

In [6]:
# ── Top scorer distribution across shadow claims ──────────────────────────────
top1_dist = df_shadow["scorer_top1_name"].value_counts().reset_index()
top1_dist.columns = ["Adjuster", "Claims Recommended"]
print("\nScorer recommendations:")
print(top1_dist.to_string(index=False))

fig = px.bar(top1_dist, x="Adjuster", y="Claims Recommended",
             title="Shadow Mode — Scorer Recommendations per Adjuster",
             color="Adjuster")
fig.show()

print(f"\nNext step: run live for 60 days, compare FTR on scorer-routed vs")
print(f"control-group claims (existing routing). Report FTR delta.")


Scorer recommendations:
      Adjuster  Claims Recommended
Rachel Coleman                  40



Next step: run live for 60 days, compare FTR on scorer-routed vs
control-group claims (existing routing). Report FTR delta.
